Question: Testing resnet 50 finer depth increments


In [1]:
import os, shutil
import keras
from training import TrainingHistory, fit_model
from training import fit_model
# from training_data import TrainingData, Normalizer
import matplotlib.pyplot as plt

2024-03-01 16:31:11.579878: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-03-01 16:31:12.822828: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
save_path = "models/test_resnet50_cont"
training_data_path = "training_data"

In [3]:
from training_data import load_training_data
training_data = load_training_data(training_data_path)
levels = list(training_data.levels.keys())
depths = [0.4, 0.3, 0.2, 0.1, 0.5]
save_paths = {depth:f"{save_path}/depth_{depth:0.02f}" for depth in depths}

In [4]:
if os.path.exists(save_path):
    shutil.rmtree(save_path)
training_level = 'd0.40'
for depth in depths:
    print(f"=== training at depth {depth} ===")
    fit_model(
        training_data=training_data_path,
        training_level=training_level,
        train_depth=depth,
        optimizer=keras.optimizers.Nadam(),
        rate_scheduler={'delay': 2, 'decay': 0.8},
        save_path=save_paths[depth],
        model_type='ResNet50',
    )

=== training at depth 0.4 ===


2024-03-01 16:31:23.350990: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1639] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22247 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:65:00.0, compute capability: 8.6


GPUS: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')] False
Loaded 995000 training examples and 25000 validation examples
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 resnet50 (Functional)       (None, 13, 13, 2048)      23587712  
                                                                 
 global_average_pooling2d (  (None, 2048)              0         
 GlobalAveragePooling2D)                                         
                                                                 
 dense (Dense)               (None, 3)                 6147      
                                                                 
Total params: 23593859 (90.00 MB)
Trainable params: 6147 (24.01 KB)
Non-trainable params: 23587712 (89.98 MB)
_________________________________________________________________
None


: 

In [ ]:
metric = 'xy_mse'
fig, ax = plt.subplots(len(levels), len(depths), figsize=(len(depths)*3, len(levels)*3), sharey=True)
fig.supylabel('validation difficulty', fontsize=20)
fig.supxlabel('training depth', fontsize=20)
for i, level in enumerate(levels):
    ax[i, 0].set_ylabel(f"{metric} loss @ {level}")
for i,depth in enumerate(depths):
    history = TrainingHistory(f'{save_paths[depth]}/training_history.json')
    history.plot_loss(metrics=[metric], ax=ax[:, i:i+1])
    ax[0, i].set_title(f"depth: {depth}")
fig.tight_layout()

In [ ]:
from model import PipetteDetectionModel
depth = 0.8
model = PipetteDetectionModel(load_model=save_paths[depth])

In [ ]:
shape = (10, len(levels))
fig, ax = plt.subplots(shape[0], shape[1], figsize=(shape[1]*3, shape[0]*3))

for j, level in enumerate(levels):
    ax[0,j].set_title(f"difficulty={level}")
    image, pip_pos = training_data.levels[level][:shape[0]].get_arrays()
    pip_pos = training_data.output_norm.denormalize(pip_pos)
    pred = training_data.output_norm.denormalize(model.model.predict(image))
    for i in range(shape[0]):
        ax[i,j].imshow(image[i], cmap='gray')
        ax[i,j].scatter(pip_pos[i, 2], pip_pos[i, 1], color='red')
        ax[i,j].scatter(pred[i, 2], pred[i, 1], color='green')
        